# 交互式查询本地 DuckLake (`fsc_lake`)

只读连接到项目里的 DuckLake catalog，方便随手 SQL 探查 bronze/silver/gold。

**运行方式**：在 VS Code 里直接打开本文件，右上角选 kernel 为 `/home/ubuntu/.venv`（已有 `ipykernel`+`pandas`+`duckdb 1.5.4`），逐格运行即可。无需安装 jupyterlab。

**要点**
- 默认 `READ_ONLY=True`：不会和 Dagster/dbt 抢 catalog 的单写锁，可以边跑管道边查。想写入时改成 `False`。
- catalog 是 DuckLake **1.0** 格式，必须 duckdb **>= 1.5**（旧版报 `Only DuckLake versions ...0.3 are supported`）。
- 所有表在 `lake.<schema>.<table>`；连接时已 `USE lake.bronze`，所以 bronze 表可直接写表名。

In [1]:
import duckdb
import polars as pl
from pathlib import Path

# --- 配置：指向包含 lake_catalog.duckdb 的项目目录 ---
BASE = Path("/home/ubuntu/dagster")
CATALOG = BASE / "lake_catalog.duckdb"
READ_ONLY = True  # 改 False 可写入（会占用 catalog 单写锁，勿与 Dagster 同时写）


def connect(read_only: bool = READ_ONLY) -> duckdb.DuckDBPyConnection:
    """打开一个已 attach 本地 DuckLake 的 duckdb 连接。"""
    con = duckdb.connect()
    con.execute("INSTALL sqlite; LOAD sqlite; INSTALL ducklake; LOAD ducklake;")
    ro = " (READ_ONLY)" if read_only else ""
    # 不传 DATA_PATH —— DuckLake 从 catalog 元数据里读，省得写错路径
    con.execute(f"ATTACH 'ducklake:sqlite:{CATALOG}' AS lake{ro}")
    con.execute("USE lake.bronze")  # 默认 schema，bronze 表可不带前缀
    return con


con = connect()
print("duckdb", duckdb.__version__, "| attached", CATALOG, "| read_only=", READ_ONLY)

duckdb 1.5.4 | attached /home/ubuntu/dagster/lake_catalog.duckdb | read_only= True


In [2]:
def q(sql: str) :
    """跑 SQL 返回 pandas DataFrame。bronze 表可直接写表名，
    其它层用全名，如 lake.silver.xxx / lake.gold.xxx。
    例: q(\"SELECT * FROM accounts LIMIT 5\")"""
    return con.execute(sql).pl()

## 1. 有哪些表

In [3]:
q("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_catalog = 'lake'
  AND table_schema NOT LIKE '%staging%'   -- 隐藏 dlt 的 *_staging
ORDER BY 1, 2
""")

table_schema,table_name
str,str
"""bronze""","""_dlt_loads"""
"""bronze""","""_dlt_pipeline_state"""
"""bronze""","""_dlt_version"""
"""bronze""","""account"""
"""bronze""","""accounts"""
"""bronze""","""certificate_c__c"""
"""bronze""","""country_data__c"""
"""bronze""","""fsc_aaf_calculationses"""
"""bronze""","""non_certificate_holder__c"""


## 2. 各 bronze 表的行数

In [4]:
tables = q("""
    SELECT table_name FROM information_schema.tables
    WHERE table_catalog='lake' AND table_schema='bronze'
      AND table_name NOT LIKE '\\_dlt%' ESCAPE '\\'   -- 跳过 dlt 内部表
    ORDER BY 1
""")["table_name"].to_list()

rows = [
    {"table": t, "rows": con.execute(f'SELECT count(*) FROM bronze."{t}"').fetchone()[0]}
    for t in tables
]
pl.DataFrame(rows)

table,rows
str,i64
"""account""",245582
"""accounts""",1026
"""certificate_c__c""",155683
"""country_data__c""",241
"""fsc_aaf_calculationses""",1520000
"""non_certificate_holder__c""",3301
"""recordtype""",46


## 3. 预览某张表

In [5]:
con.execute("SELECT * FROM fsc_aaf_calculationses LIMIT 20").pl()

@odata.etag,fsc_aaf_aaffmamount@OData.Community.Display.V1.FormattedValue,fsc_aaf_aaffmamount,fsc_aaf_dateto@OData.Community.Display.V1.FormattedValue,fsc_aaf_dateto,statuscode@OData.Community.Display.V1.FormattedValue,statuscode,fsc_aaf_sfid,fsc_aaf_isrecordcreated@OData.Community.Display.V1.FormattedValue,fsc_aaf_isrecordcreated,fsc_aaf_firstissuedate@OData.Community.Display.V1.FormattedValue,fsc_aaf_firstissuedate,_fsc_aaf_country_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_country_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_country_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_country_value,_ownerid_value@OData.Community.Display.V1.FormattedValue,_ownerid_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_ownerid_value@Microsoft.Dynamics.CRM.lookuplogicalname,_ownerid_value,fsc_aaf_fullcertificatecode,fsc_aaf_triggerprocess@OData.Community.Display.V1.FormattedValue,fsc_aaf_triggerprocess,fsc_aaf_cb,_fsc_aaf_batch_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_batch_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_batch_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_batch_value,fsc_aaf_cert_type@OData.Community.Display.V1.FormattedValue,fsc_aaf_cert_type,_fsc_aaf_method_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_method_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_method_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_method_value,fsc_aaf_isaafexempted@OData.Community.Display.V1.FormattedValue,fsc_aaf_isaafexempted,_createdby_value@OData.Community.Display.V1.FormattedValue,…,fsc_aaf_slimftype@OData.Community.Display.V1.FormattedValue,fsc_aaf_slimftype,fsc_aaf_area_slimfplantation@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfplantation,fsc_aaf_area_slimfconversation@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfconversation,fsc_aaf_area_slimftemperate@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimftemperate,fsc_aaf_area_slimfcommunity@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfcommunity,fsc_aaf_area_community@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_community,fsc_aaf_area_tropical@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_tropical,fsc_aaf_area_boreal@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_boreal,fsc_aaf_area_slimfboreal@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfboreal,fsc_aaf_area_slimftropical@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimftropical,_fsc_aaf_projectcertificate_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_projectcertificate_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_projectcertificate_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_projectcertificate_value,fsc_aaf_pcstatus@OData.Community.Display.V1.FormattedValue,fsc_aaf_pcstatus,fsc_aaf_pcname,fsc_aaf_pccode,fsc_aaf_projectcertificateamount@OData.Community.Display.V1.FormattedValue,fsc_aaf_projectcertificateamount,fsc_aaf_pctype@OData.Community.Display.V1.FormattedValue,fsc_aaf_pctype,fsc_aaf_projectcertificationdate@OData.Community.Display.V1.FormattedValue,fsc_aaf_projectcertificationdate,fsc_aaf_pcaafclass@OData.Community.Display.V1.FormattedValue,fsc_aaf_pcaafclass,fsc_aaf_project_cost
str,str,f64,str,"datetime[μs, Europe/Berlin]",str,i64,str,str,bool,str,"datetime[μs, Europe/Berlin]",str,str,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,i64,str,str,str,str,str,bool,str,…,str,str,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,str,str,str,str,i64,str,str,str,f64,str,i64,str,"datetime[μs, Europe/Berlin]",str,i64,str
"""W/""1846817674""""","""824""",824.0,"""31.05.2029""",2029-05-31 02:00:00 CEST,"""Active""",1,"""https://na151.salesforce.com/a…","""No""",false,"""31.05.1999""",1999-05-31 20:30:00 CEST,"""UNITED KINGDOM""","""fsc_aaf_country""","""fsc_countrydata""","""991f8543-a036-ea11-a812-000d3a…","""Con

In [7]:
q("SELECT * FROM fsc_aaf_calculationses LIMIT 20")

,@odata.etag,fsc_aaf_aaffmamount@OData.Community.Display.V1.FormattedValue,fsc_aaf_aaffmamount,fsc_aaf_dateto@OData.Community.Display.V1.FormattedValue,fsc_aaf_dateto,statuscode@OData.Community.Display.V1.FormattedValue,statuscode,fsc_aaf_sfid,fsc_aaf_isrecordcreated@OData.Community.Display.V1.FormattedValue,fsc_aaf_isrecordcreated,...,fsc_aaf_pccode,fsc_aaf_projectcertificateamount@OData.Community.Display.V1.FormattedValue,fsc_aaf_projectcertificateamount,fsc_aaf_pctype@OData.Community.Display.V1.FormattedValue,fsc_aaf_pctype,fsc_aaf_projectcertificationdate@OData.Community.Display.V1.FormattedValue,fsc_aaf_projectcertificationdate,fsc_aaf_pcaafclass@OData.Community.Display.V1.FormattedValue,fsc_aaf_pcaafclass,fsc_aaf_project_cost
0,"W/""1846817674""",824,824.0,31.05.2029,2029-05-31 02:00:00+02:00,Active,1,https://na151.salesforce.com/a0240000005sVhiAAE,No,False,...,None,None,NaN,None,<NA>,None,NaT,None,<NA>,None
1,"W/""1847052886""",None,NaN,None,NaT,Active,1,https://na151.salesforce.com/a0IUV00000FZz812AD,No,False,...,CU-PRO-863127,358,358.0,Partial Certification,1,14.03.2026,2026-03-14 19:30:00+01:00,Trader Class 2,2,3149165
2,"W/""1847200764""",None,NaN,None,NaT,Active,1,https://na151.salesforce.com/a0IUV00000CCUzV2AX,Yes,True,...,TT-PRO-008152,323,323.0,Partial Certification,1,22.06.2026,2026-06-22 02:00:00+02:00,Trader Class 2,2,2646060
3,"W/""1847203429""",0,0.0,05.07.2027,2027-07-05 02:00:00+02:00,Active,1,https://na151.salesforce.com/a024y00000jqbXIAAY,Yes,True,...,None,None,NaN,None,<NA>,None,NaT,None,<NA>,None
4,"W/""1847203441""",0,0.0,20.01.2027,2027-01-20 01:00:00+01:00,Active,1,https://na151.salesforce.com/a0240000006uaaRAAQ,Yes,True,...,None,None,NaN,None,<NA>,None,NaT,None,<NA>,None
5,"W/""1847203459""",0,0.0,03.02.2028,2028-02-03 01:00:00+01:00,Active,1,https://na151.salesforce.com/a0240000005sUjeAAE,Yes,True,...,None,None,NaN,None,<NA>,None,NaT,None,<NA>,None
6,"W/""1847203477""",0,0.0,15.02.2029,2029-02-15 01:00:00+01:00,Active,1,https://na151.salesforce.com/a0240000005sTDdAAM,Yes,True,...,None,None,NaN,None,<NA>,None,NaT,None,<NA>,None
7,"W/""1847203504""",0,0.0,30.05.2027,2027-05-30 02:00:00+02:00,Active,1,https://na151.salesforce.com/a0240000005sR6PAAU,Yes,True,...,None,None,NaN,None,<NA>,None,NaT,None,<NA>,None
8,"W/""1847203512""",0,0.0,29.05.2027,2027-05-29 02:00:00+02:00,Active,1,https://na151.salesforce.com/a0240000005sVB5AAM,Yes,True,...,None,None,NaN,None,<NA>,None,NaT,None,<NA>,None
9,"W/""1847203518""",0,0.0,06.03.2027,2027-03-06 01:00:00+01:00,Active,1,https://na151.salesforce.com/a0240000005sShoAAE,Yes,True,...,None,None,NaN,None,<NA>,None,NaT,None,<NA>,None


In [6]:
import polars as pl

In [6]:
pl.read_parquet("/home/ubuntu/dagster/lake/bronze/fsc_aaf_calculationses/ducklake-019f64a8-99f5-7ea7-89ee-c978ffb1795d.parquet")

@odata.etag,fsc_aaf_aaffmamount@OData.Community.Display.V1.FormattedValue,fsc_aaf_aaffmamount,fsc_aaf_dateto@OData.Community.Display.V1.FormattedValue,fsc_aaf_dateto,statuscode@OData.Community.Display.V1.FormattedValue,statuscode,fsc_aaf_sfid,fsc_aaf_isrecordcreated@OData.Community.Display.V1.FormattedValue,fsc_aaf_isrecordcreated,fsc_aaf_firstissuedate@OData.Community.Display.V1.FormattedValue,fsc_aaf_firstissuedate,_fsc_aaf_country_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_country_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_country_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_country_value,_ownerid_value@OData.Community.Display.V1.FormattedValue,_ownerid_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_ownerid_value@Microsoft.Dynamics.CRM.lookuplogicalname,_ownerid_value,fsc_aaf_fullcertificatecode,fsc_aaf_triggerprocess@OData.Community.Display.V1.FormattedValue,fsc_aaf_triggerprocess,fsc_aaf_cb,_fsc_aaf_batch_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_batch_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_batch_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_batch_value,fsc_aaf_cert_type@OData.Community.Display.V1.FormattedValue,fsc_aaf_cert_type,_fsc_aaf_method_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_method_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_method_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_method_value,fsc_aaf_isaafexempted@OData.Community.Display.V1.FormattedValue,fsc_aaf_isaafexempted,_createdby_value@OData.Community.Display.V1.FormattedValue,…,fsc_aaf_slimftype@OData.Community.Display.V1.FormattedValue,fsc_aaf_slimftype,fsc_aaf_area_slimfplantation@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfplantation,fsc_aaf_area_slimfconversation@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfconversation,fsc_aaf_area_slimftemperate@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimftemperate,fsc_aaf_area_slimfcommunity@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfcommunity,fsc_aaf_area_community@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_community,fsc_aaf_area_tropical@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_tropical,fsc_aaf_area_boreal@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_boreal,fsc_aaf_area_slimfboreal@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimfboreal,fsc_aaf_area_slimftropical@OData.Community.Display.V1.FormattedValue,fsc_aaf_area_slimftropical,_fsc_aaf_projectcertificate_value@OData.Community.Display.V1.FormattedValue,_fsc_aaf_projectcertificate_value@Microsoft.Dynamics.CRM.associatednavigationproperty,_fsc_aaf_projectcertificate_value@Microsoft.Dynamics.CRM.lookuplogicalname,_fsc_aaf_projectcertificate_value,fsc_aaf_pcstatus@OData.Community.Display.V1.FormattedValue,fsc_aaf_pcstatus,fsc_aaf_pcname,fsc_aaf_pccode,fsc_aaf_projectcertificateamount@OData.Community.Display.V1.FormattedValue,fsc_aaf_projectcertificateamount,fsc_aaf_pctype@OData.Community.Display.V1.FormattedValue,fsc_aaf_pctype,fsc_aaf_projectcertificationdate@OData.Community.Display.V1.FormattedValue,fsc_aaf_projectcertificationdate,fsc_aaf_pcaafclass@OData.Community.Display.V1.FormattedValue,fsc_aaf_pcaafclass,fsc_aaf_project_cost
str,str,f64,str,"datetime[μs, UTC]",str,i64,str,str,bool,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,i64,str,str,str,str,str,bool,str,…,str,str,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,f64,str,str,str,str,str,i64,str,str,str,f64,str,i64,str,"datetime[μs, UTC]",str,i64,str
"""W/""1846817674""""","""824""",824.0,"""31.05.2029""",2029-05-31 00:00:00 UTC,"""Active""",1,"""https://na151.salesforce.com/a…","""No""",false,"""31.05.1999""",1999-05-31 18:30:00 UTC,"""UNITED KINGDOM""","""fsc_aaf_country""","""fsc_countrydata""","""991f8543-a036-ea11-a812-000d3a…","""Congruent Support1""","""ownerid"""

## 4. 看某张表的列 / 类型

In [9]:
q("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_catalog='lake' AND table_schema='bronze'
  AND table_name = 'fsc_aaf_calculationses'
ORDER BY ordinal_position
""")

column_name,data_type
str,str
"""@odata.etag""","""VARCHAR"""
"""fsc_aaf_aaffmamount@OData.Comm…","""VARCHAR"""
"""fsc_aaf_aaffmamount""","""DOUBLE"""
"""fsc_aaf_dateto@OData.Community…","""VARCHAR"""
"""fsc_aaf_dateto""","""TIMESTAMP WITH TIME ZONE"""
…,…
"""fsc_aaf_projectcertificationda…","""VARCHAR"""
"""fsc_aaf_projectcertificationda…","""TIMESTAMP WITH TIME ZONE"""
"""fsc_aaf_pcaafclass@OData.Commu…","""VARCHAR"""


## 5. 你自己的查询（scratch）

In [ ]:
q("""
-- 在这里写你的 SQL
SELECT count(*) AS n
FROM fsc_aaf_calculationses
""")

## 6. DuckLake 特性：快照 & 时间旅行

DuckLake 每次写入都产生一个 snapshot，可以查历史版本。

In [9]:
# 最近 10 个快照
q("SELECT snapshot_id, snapshot_time, schema_version FROM ducklake_snapshots('lake') ORDER BY snapshot_id DESC LIMIT 10")

,snapshot_id,snapshot_time,schema_version
0,59,2026-07-15 09:29:56.380330+02:00,16
1,58,2026-07-15 09:29:56.137099+02:00,16
2,57,2026-07-15 09:29:55.628920+02:00,16
3,56,2026-07-15 09:29:55.326002+02:00,16
4,55,2026-07-15 09:29:55.030388+02:00,16
5,54,2026-07-15 09:29:54.907654+02:00,16
6,53,2026-07-15 09:29:54.715824+02:00,15
7,52,2026-07-15 09:23:20.635998+02:00,14
8,51,2026-07-15 09:23:18.456160+02:00,14
9,50,2026-07-15 09:23:17.592008+02:00,14


In [ ]:
# 时间旅行：读某个历史快照时的表内容（把 42 换成上面的 snapshot_id）
# q("SELECT count(*) FROM accounts AT (VERSION => 42)")

## 7. 收尾

In [ ]:
con.close()
print("connection closed")